In [1]:
from datetime import datetime, timedelta
import json
import uuid
import random 
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import cultpass

# Udahub Accounts

## Cultpass Database

**Init DB**

In [2]:
cultpass_db = "data/external/cultpass.db"

In [3]:
reset_db(cultpass_db)

✅ Removed existing data/external/cultpass.db
2026-11-11 22:57:59,800 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026-11-11 22:57:59,800 INFO sqlalchemy.engine.Engine COMMIT


✅ Recreated data/external/cultpass.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{cultpass_db}", echo=False)
cultpass.Base.metadata.create_all(engine)

**Experiences**

In [5]:
experience_data = []

with open('data/external/cultpass_experiences.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        experience_data.append(json.loads(line))

In [6]:
experience_data

[{'title': 'Carnival History Tour in Olinda',
  'description': "Discover the origins and vibrant traditions of Pernambuco's Carnival.",
  'location': 'Pernambuco, Brazil'},
 {'title': 'Sunset Paddleboarding',
  'description': 'Glide across calm waters at golden hour with all gear included.',
  'location': 'Santa Catarina, Brazil'},
 {'title': 'Pelourinho Colonial Walk',
  'description': 'Wander through colorful streets and learn about Afro-Brazilian history.',
  'location': 'Bahia, Brazil'},
 {'title': 'Samba Night at Lapa',
  'description': 'Dance the night away at a traditional samba club in the Lapa arches.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Christ the Redeemer Experience',
  'description': 'Take a guided trip to one of the New Seven Wonders of the World with historical context.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Modern Art at MASP',
  'description': 'Enjoy a guided visit to the São Paulo Museum of Art with insights into its top collections.',

In [7]:
with get_session(engine) as session:
    experiences = []

    for idx, experience in enumerate(experience_data):
        exp = cultpass.Experience(
            experience_id=str(uuid.uuid4())[:6],
            title=experience["title"],
            description=experience["description"],
            location=experience["location"],
            when=datetime.now() + timedelta(days=idx+1),
            slots_available=random.randint(1,30),
            is_premium=(idx % 2 == 0)
        )
        experiences.append(exp)

    session.add_all(experiences)

**User**

In [8]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [9]:
cultpass_users

[{'id': 'a4ab87',
  'name': 'Alice Kingsley',
  'email': 'alice.kingsley@wonderland.com',
  'is_blocked': True},
 {'id': 'f556c0',
  'name': 'Bob Stone',
  'email': 'bob.stone@granite.com',
  'is_blocked': False},
 {'id': '88382b',
  'name': 'Cathy Bloom',
  'email': 'cathy.bloom@florals.org',
  'is_blocked': False},
 {'id': '888fb2',
  'name': 'David Noir',
  'email': 'david.noir@shadowmail.com',
  'is_blocked': True},
 {'id': 'f1f10d',
  'name': 'Eva Green',
  'email': 'eva.green@ecosoul.net',
  'is_blocked': False},
 {'id': 'e6376d',
  'name': 'Frank Ocean',
  'email': 'frank.ocean@seawaves.io',
  'is_blocked': False}]

In [10]:
with get_session(engine) as session:
    db_users = []
    for user_info in cultpass_users:
        user = cultpass.User(
            user_id=user_info["id"],
            full_name=user_info["name"],
            email=user_info["email"],
            is_blocked=user_info["is_blocked"],
            created_at=datetime.now()
        )
        db_users.append(user)
    session.add_all(db_users) 

**Subscription**

In [11]:
with get_session(engine) as session:
    subscriptions = []
    for user_info in cultpass_users:
        subscription = cultpass.Subscription(
            subscription_id=str(uuid.uuid4())[:6],
            user_id=user_info["id"],
            status=random.choice(["active", "cancelled"]),
            tier=random.choice(["basic", "premium"]),
            monthly_quota=random.randint(2,10),
            started_at=datetime.now()
        )
        subscriptions.append(subscription)

    session.add_all(subscriptions)

**Reservation**

In [12]:
# Applicable to `cultpass_users[0]` at the moment

with get_session(engine) as session:
    experience_ids = [
        exp.experience_id 
        for exp 
        in session.query(cultpass.Experience).all()
    ]

    reservation1 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    reservation2 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    session.add_all([reservation1, reservation2])

In [13]:
# Deterministic demo data for the UDA-Hub scenarios.
# Subscriptions above are random, so pin the ones the demos rely on,
# and give more users reservations (the starter only gave reservations to the first user).
demo_subscriptions = {
    "f556c0": {"status": "active", "tier": "premium", "monthly_quota": 6},    # Bob Stone
    "88382b": {"status": "active", "tier": "basic", "monthly_quota": 4},      # Cathy Bloom
    "f1f10d": {"status": "cancelled", "tier": "basic", "monthly_quota": 4},   # Eva Green
    "e6376d": {"status": "active", "tier": "premium", "monthly_quota": 2},    # Frank Ocean
}

with get_session(engine) as session:
    for user_id, values in demo_subscriptions.items():
        sub = session.query(cultpass.Subscription).filter_by(user_id=user_id).one()
        for key, value in values.items():
            setattr(sub, key, value)

    experiences = session.query(cultpass.Experience).order_by(cultpass.Experience.when).all()
    for exp in experiences:
        exp.slots_available = max(exp.slots_available, 5)

    standard = [e for e in experiences if not e.is_premium]
    premium = [e for e in experiences if e.is_premium]
    demo_reservations = [
        ("f556c0", premium[0]),   # Bob: premium experience, tomorrow
        ("f556c0", standard[1]),  # Bob: standard experience
        ("88382b", standard[0]),  # Cathy
        ("e6376d", premium[1]),   # Frank: uses 1 of 2 credits
    ]
    session.add_all([
        cultpass.Reservation(
            reservation_id=str(uuid.uuid4())[:6],
            user_id=user_id,
            experience_id=exp.experience_id,
            status="reserved",
        )
        for user_id, exp in demo_reservations
    ])

# Tests

In [14]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user)

<User(user_id='a4ab87', email='alice.kingsley@wonderland.com', is_blocked=True)>
<User(user_id='f556c0', email='bob.stone@granite.com', is_blocked=False)>
<User(user_id='88382b', email='cathy.bloom@florals.org', is_blocked=False)>
<User(user_id='888fb2', email='david.noir@shadowmail.com', is_blocked=True)>
<User(user_id='f1f10d', email='eva.green@ecosoul.net', is_blocked=False)>
<User(user_id='e6376d', email='frank.ocean@seawaves.io', is_blocked=False)>


In [15]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user.subscription)

<Subscription(subscription_id='df75b2', user_id='a4ab87', status='active', tier='premium')>
<Subscription(subscription_id='9dfcb3', user_id='f556c0', status='active', tier='premium')>
<Subscription(subscription_id='6dfc28', user_id='88382b', status='active', tier='basic')>
<Subscription(subscription_id='6c32f9', user_id='888fb2', status='cancelled', tier='premium')>
<Subscription(subscription_id='d6ff12', user_id='f1f10d', status='cancelled', tier='basic')>
<Subscription(subscription_id='09ab10', user_id='e6376d', status='active', tier='premium')>


In [16]:
with get_session(engine) as session:
    experiences = session.query(cultpass.Experience).all()
    for experience in experiences:
        print(experience)

<Experience(experience_id='b13789', title='Carnival History Tour in Olinda', when='2026-11-12 22:57:59.816156')>
<Experience(experience_id='a05fe0', title='Sunset Paddleboarding', when='2026-11-13 22:57:59.819110')>
<Experience(experience_id='859c7c', title='Pelourinho Colonial Walk', when='2026-11-14 22:57:59.819139')>
<Experience(experience_id='b0e4dd', title='Samba Night at Lapa', when='2026-11-15 22:57:59.819151')>
<Experience(experience_id='32ff03', title='Christ the Redeemer Experience', when='2026-11-16 22:57:59.819162')>
<Experience(experience_id='8aaed6', title='Modern Art at MASP', when='2026-11-17 22:57:59.819171')>
<Experience(experience_id='47fae6', title='Ibirapuera Park Bike Ride', when='2026-11-18 22:57:59.819181')>


In [17]:
with get_session(engine) as session:
    for reservation in session.query(cultpass.Reservation).all():
        print(reservation, "|", reservation.experience.title)

<Reservation(reservation_id='9692e6', user_id='a4ab87', experience_id='32ff03', status='reserved')> | Christ the Redeemer Experience
<Reservation(reservation_id='93dfaf', user_id='a4ab87', experience_id='32ff03', status='reserved')> | Christ the Redeemer Experience
<Reservation(reservation_id='ef6a30', user_id='f556c0', experience_id='b13789', status='reserved')> | Carnival History Tour in Olinda
<Reservation(reservation_id='b62296', user_id='f556c0', experience_id='b0e4dd', status='reserved')> | Samba Night at Lapa
<Reservation(reservation_id='ea9f9a', user_id='88382b', experience_id='a05fe0', status='reserved')> | Sunset Paddleboarding
<Reservation(reservation_id='811e05', user_id='e6376d', experience_id='859c7c', status='reserved')> | Pelourinho Colonial Walk
